# Desafio 3 - ZettaLab

## Engenharia de Features

### Importação de Bibliotecas

In [1]:
import pandas as pd             # Biblioteca para manipulação e análise de dados
import numpy as np              # Biblioteca para calculo de vetores e matrizes
import sys                      # Biblioteca para acessar variaveis e funções que interagem fortemente com o interpretador
import os                       # Biblioteca para interação com arquivos e diretórios a nivel sistema operacional
import basedosdados as bd       # Biblioteca para acessar o datalake público do site BasedosDados
import glob                     # Biblioteca para processamento em lote
from pathlib import Path        # Biblioteca para a manipulação de caminhos do sistema a nivel orientado a objetos
import gzip                     # Biblioteca para compressão e decompressão de dados usando o formato gzip
import shutil                   # Biblioteca para manipulação de arquivos e diretorios
import re

current_dir = os.getcwd()
parent_dir = os.path.abspath(os.path.join(current_dir, '..'))
if parent_dir not in sys.path:
    sys.path.append(parent_dir)
    
import config_path          # Módulo que salva todos os caminhos de diretórios utilizados no projeto

from scripts import features        # Módulo de criação e manipulação de features
from scripts import modeling        # Módulo de modelagem de IA
from scripts import utils           # Módulo de utilitários genéricos
from scripts import pre_processing  # Módulo de pré-processamento e obtenção de dados.

In [2]:
df_uf_cods = pd.read_csv(config_path.PROCESSED_DATA_DIRECTORY_PATH / "codigos_municipios.csv")
df_mun_cods = pd.read_csv(config_path.PROCESSED_DATA_DIRECTORY_PATH / "codigos_ufs.csv")
df_mun_area = pd.read_csv(config_path.PROCESSED_DATA_DIRECTORY_PATH / "municipio_area_2024.csv")
df_uf_area = pd.read_csv(config_path.PROCESSED_DATA_DIRECTORY_PATH / "uf_area_2024.csv")
df_rg_area = pd.read_csv(config_path.PROCESSED_DATA_DIRECTORY_PATH / "rg_area_2024.csv")
df_inmet_2025 = pd.read_parquet(config_path.PROCESSED_DATA_DIRECTORY_PATH / "df_inmet_2025.parquet")
df_monitor_fogo = pd.read_csv(config_path.PROCESSED_DATA_DIRECTORY_PATH / "monitor-fogo-mapbiomas.csv")
df_dados_BDMEP = pd.read_parquet(config_path.PROCESSED_DATA_DIRECTORY_PATH / "dados_meteorologicos_BDMEP_processado.parquet")
df_banco_queimadas = pd.read_parquet(config_path.PROCESSED_DATA_DIRECTORY_PATH / "banco_dados_queimadas.parquet")

In [3]:
df_monitor_fogo

,Ano,Bioma,Estado,Mes_nome,Mes_num,Categoria_principal,Tipo_uso,Subtipo_uso,Subclasse,Classe_fina,Classe_final,Area_queimada_ha
0,2019,Amazônia,Acre,Abril,4,Antrópico,Agropecuária,Pastagem,Pastagem,Pastagem,Pastagem,11.043091
1,2019,Amazônia,Acre,Abril,4,Natural,Floresta,Formação Florestal,Formação Florestal,Formação Florestal,Formação Florestal,0.177260
2,2019,Amazônia,Acre,Agosto,8,Antrópico,Agropecuária,Agricultura,Agricultura,Lavoura Temporária,Outras Lavouras Temporárias,46.909020
3,2019,Amazônia,Acre,Agosto,8,Antrópico,Agropecuária,Pastagem,Pastagem,Pastagem,Pastagem,66041.145360
4,2019,Amazônia,Acre,Agosto,8,Natural,Floresta,Formação Florestal,Floresta Alagável,Floresta Alagável,Floresta Alagável,68.431673
...,...,...,...,...,...,...,...,...,...,...,...,...
25075,2024,Pantanal,Mato Grosso do Sul,Março,3,Antrópico,Corpos D´água,Outros,"Rios, Lagos e Oceano","Rios, Lagos e Oceano","Rios, Lagos e Oceano",381.057779
25076,2024,Pantanal,Mato Grosso do Sul,Março,3,Natural,Floresta,Formação Florestal,Formação Florestal,Formação Florestal,Formação Florestal,433.061518
25077,2024,Pantanal,Mato Grosso do Sul,Março,3,Natural,Floresta,Formação Savânica,Formação Savânica,Formação Savânica,Formação Savânica,655.863783
25078,2024,Pantanal,Mato Grosso do Sul,Março,3,Natural,Formação Natural não Florestal,Campo Alagado e Área Pantanosa,Campo Alagado e Área Pantanosa,Campo Alagado e Área Pantanosa,Campo Alagado e Área Pantanosa,2370.983449


In [4]:
df_monitor_fogo = df_monitor_fogo.sort_values(by=["Estado","Bioma", "Classe_final", "Ano", "Mes_num"])

# Criar coluna de data (útil pra séries temporais)
df_monitor_fogo["Data"] = pd.to_datetime(df_monitor_fogo["Ano"].astype(str) + "-" + df_monitor_fogo["Mes_num"].astype(str) + "-01")

# Area queimada (variação mensal)
df_monitor_fogo["delta_area"] = df_monitor_fogo.groupby(
    ["Estado", "Bioma", "Classe_final"]
)["Area_queimada_ha"].diff()

# Taxa de crescimento
df_monitor_fogo["taxa_crescimento"] = df_monitor_fogo.groupby(
    ["Estado", "Bioma", "Classe_final"]
)["Area_queimada_ha"].pct_change()

# Aceleração (segunda derivada)
df_monitor_fogo["aceleracao"] = df_monitor_fogo.groupby(
    ["Estado", "Bioma", "Classe_final"]
)["delta_area"].diff()

# Média móvel (3 meses)
df_monitor_fogo["media_movel_3"] = df_monitor_fogo.groupby(
    ["Estado", "Bioma", "Classe_final"]
)["Area_queimada_ha"].transform(lambda x: x.rolling(3).mean())

# Desvio padrão (volatilidade)
df_monitor_fogo["volatilidade_6m"] = df_monitor_fogo.groupby(
    ["Estado", "Bioma", "Classe_final"]
)["Area_queimada_ha"].transform(lambda x: x.rolling(6).std())

# % dentro do estado por mês
total_estado_mes = df_monitor_fogo.groupby(
    ["Estado", "Ano", "Mes_num"]
)["Area_queimada_ha"].transform("sum")

df_monitor_fogo["pct_estado"] = df_monitor_fogo["Area_queimada_ha"] / total_estado_mes

# % dentro do bioma
total_bioma = df_monitor_fogo.groupby(
    ["Bioma", "Ano", "Mes_num"]
)["Area_queimada_ha"].transform("sum")

df_monitor_fogo["pct_bioma"] = df_monitor_fogo["Area_queimada_ha"] / total_bioma

# Pressão por tipo de uso
total_uso = df_monitor_fogo.groupby(
    ["Estado", "Ano", "Mes_num"]
)["Area_queimada_ha"].transform("sum")

df_monitor_fogo["pressao_uso"] = df_monitor_fogo["Area_queimada_ha"] / total_uso

# Ranking de impacto por classe
df_monitor_fogo["rank_classe"] = df_monitor_fogo.groupby(
    ["Estado", "Ano", "Mes_num"]
)["Area_queimada_ha"].rank(ascending=False)

# Area acumulado no ano
df_monitor_fogo["area_acumulada_ano"] = df_monitor_fogo.groupby(
    ["Estado", "Bioma", "Classe_final", "Ano"]
)["Area_queimada_ha"].cumsum()

# Area acumulada total (historico)
df_monitor_fogo["area_acumulada_total"] = df_monitor_fogo.groupby(
    ["Estado", "Bioma", "Classe_final"]
)["Area_queimada_ha"].cumsum()

df_monitor_fogo.replace([np.inf, -np.inf], np.nan, inplace=True)

In [5]:
df_monitor_fogo

,Ano,Bioma,Estado,Mes_nome,Mes_num,Categoria_principal,Tipo_uso,Subtipo_uso,Subclasse,Classe_fina,...,taxa_crescimento,aceleracao,media_movel_3,volatilidade_6m,pct_estado,pct_bioma,pressao_uso,rank_classe,area_acumulada_ano,area_acumulada_total
21,2019,Amazônia,Acre,Junho,6,Natural,Formação Natural não Florestal,Campo Alagado e Área Pantanosa,Campo Alagado e Área Pantanosa,Campo Alagado e Área Pantanosa,...,NaN,NaN,NaN,NaN,0.002138,0.000005,0.002138,4.0,1.056816,1.056816
16,2019,Amazônia,Acre,Julho,7,Natural,Formação Natural não Florestal,Campo Alagado e Área Pantanosa,Campo Alagado e Área Pantanosa,Campo Alagado e Área Pantanosa,...,6.333567,NaN,NaN,NaN,0.003425,0.000024,0.003425,5.0,8.807046,8.807046
7,2019,Amazônia,Acre,Agosto,8,Natural,Formação Natural não Florestal,Campo Alagado e Área Pantanosa,Campo Alagado e Área Pantanosa,Campo Alagado e Área Pantanosa,...,9.784724,69.140445,30.797045,NaN,0.001140,0.000048,0.001140,4.0,92.391135,92.391135
44,2019,Amazônia,Acre,Setembro,9,Natural,Formação Natural não Florestal,Campo Alagado e Área Pantanosa,Campo Alagado e Área Pantanosa,Campo Alagado e Área Pantanosa,...,-0.195034,-92.135634,52.872212,NaN,0.000697,0.000037,0.000697,4.0,159.673450,159.673450
37,2019,Amazônia,Acre,Outubro,10,Natural,Formação Natural não Florestal,Campo Alagado e Área Pantanosa,Campo Alagado e Área Pantanosa,Campo Alagado e Área Pantanosa,...,-0.803338,-37.748671,54.699425,NaN,0.001224,0.000013,0.001224,4.0,172.905320,172.905320
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23909,2024,Cerrado,Tocantins,Abril,4,Antrópico,Agropecuária,Agricultura,Agricultura,Lavoura Temporária,...,-0.588197,-277.842757,124.015658,55.558195,0.031064,0.002061,0.031064,6.0,463.917762,83650.357680
23982,2024,Cerrado,Tocantins,Maio,5,Antrópico,Agropecuária,Agricultura,Agricultura,Lavoura Temporária,...,35.370536,3282.783943,1183.470107,1281.098421,0.021793,0.011374,0.021793,7.0,3708.489381,86894.929298
23970,2024,Cerrado,Tocantins,Junho,6,Antrópico,Agropecuária,Agricultura,Agricultura,Lavoura Temporária,...,-0.744229,-5570.067206,1387.882550,1253.064642,0.003620,0.001553,0.003620,8.0,4538.356619,87724.796536
23956,2024,Cerrado,Tocantins,Julho,7,Antrópico,Agropecuária,Agricultura,Agricultura,Lavoura Temporária,...,1.308022,3500.188620,1996.596778,1280.489790,0.007049,0.002796,0.007049,8.0,6453.708097,89640.148014


In [6]:
df_monitor_fogo.to_csv(config_path.FEATURES_DIRECTORY_PATH / "df_monitor_fogo_features.csv", index=False)

In [7]:
df_inmet_2025

,data,hora_utc,precip_total_mm,pressao_atms_nivel_estacao_mB,pressao_atms_maximo_na_hora_mB,pressao_atms_minimo_na_hora_mB,radiacao_global_kj_m,temp_ar_bulbo_seco_C,tempo_ponto_orvarlho_C,temp_max_hora_C,...,hora,data_hora,REGIAO,UF,ESTACAO,CODIGO (WMO),LATITUDE,LONGITUDE,ALTITUDE,DATA DE FUNDACAO
0,2025/01/01,0000 UTC,0.0,886.1,886.1,885.5,1.7,20.8,19.5,20.9,...,0000,2025-01-01 00:00:00,CO,DF,BRASILIA,A001,"-15,78944444","-47,92583332","1160,96",07/05/00
1,2025/01/01,0000 UTC,0.0,919.5,919.6,919.3,39.7,20.5,19.9,20.7,...,0000,2025-01-01 00:00:00,SE,MG,RIO PARDO DE MINAS,A551,"-15,72305554","-42,43583333","850,06",17/11/07
2,2025/01/01,0000 UTC,0.0,982.6,982.6,982.2,12.6,23.5,22.0,24.2,...,0000,2025-01-01 00:00:00,SE,MG,ITAOBIM,A550,"-16,57555554","-41,48555555","271,63",05/09/07
3,2025/01/01,0000 UTC,0.0,929.4,929.4,929.2,35.3,20.6,19.8,20.9,...,0000,2025-01-01 00:00:00,SE,MG,AGUAS VERMELHAS,A549,"-15,75166666","-41,45777777","754,07",09/09/07
4,2025/01/01,0000 UTC,0.0,918.4,918.4,917.8,1003.0,24.0,20.6,24.1,...,0000,2025-01-01 00:00:00,SE,MG,CHAPADA GAUCHA,A548,"-15,30027777","-45,61749999","873,2",22/06/07
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5020243,2025/12/31,2300 UTC,0.0,1004.8,1004.8,1004.1,2.1,27.2,23.3,28.1,...,2300,2025-12-31 23:00:00,N,AM,EIRUNEPE,A109,"-6,65027777","-69,8686111","121,54",31/08/12
5020244,2025/12/31,2300 UTC,0.0,996.8,996.8,996.0,40.7,27.3,23.5,28.6,...,2300,2025-12-31 23:00:00,N,AM,BOCA DO ACRE,A110,"-8,77666666","-67,33249999","111,98",15/07/08
5020245,2025/12/31,2300 UTC,0.0,1001.6,1003.3,1001.6,2850.5,30.4,22.8,31.0,...,2300,2025-12-31 23:00:00,N,AM,LABREA,A111,"-7,26055555","-64,7886111","61,91",27/07/08
5020246,2025/12/31,2300 UTC,0.0,1001.6,1003.3,1001.6,2850.5,30.4,22.8,31.0,...,2300,2025-12-31 23:00:00,N,AM,APUI,A113,"-7,20555555","-59,8886111","156,97",19/07/08


In [8]:
df_banco_queimadas

,data_hora,bioma,Sigla_UF,Nome_UF,ID_Município,Nome_Município,latitude,longitude,satelite,dias_sem_chuva,precipitacao,risco_fogo,potencia_radiativa_fogo,ID_UF,Data,Hora,Ano,Mes,Dia
0,2024-11-12 16:58:00,Cerrado,MG,Minas Gerais,3100104,Abadia dos Dourados,-18.38586,-47.36140,NOAA-20,2.0,12.71,0.00,3.1,31,2024-11-12,16:58:00,2024,11,12
1,2024-11-13 04:08:00,Cerrado,MG,Minas Gerais,3100104,Abadia dos Dourados,-18.37368,-47.51899,NOAA-20,2.0,12.85,0.00,1.4,31,2024-11-13,04:08:00,2024,11,13
2,2024-11-13 04:08:00,Cerrado,MG,Minas Gerais,3100104,Abadia dos Dourados,-18.37182,-47.51684,NOAA-20,2.0,12.80,0.00,3.0,31,2024-11-13,04:08:00,2024,11,13
3,2024-11-20 17:24:00,Cerrado,MG,Minas Gerais,3100104,Abadia dos Dourados,-18.35582,-47.42835,NPP-375,1.0,4.03,0.02,2.8,31,2024-11-20,17:24:00,2024,11,20
4,2024-11-20 17:24:00,Cerrado,MG,Minas Gerais,3100104,Abadia dos Dourados,-18.35275,-47.42649,NPP-375,1.0,4.00,0.02,4.2,31,2024-11-20,17:24:00,2024,11,20
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
377828,2024-10-30 17:39:00,Mata Atlântica,MG,Minas Gerais,3169406,Três Pontas,-21.36878,-45.39243,AQUA_M-T,0.0,0.30,0.03,18.8,31,2024-10-30,17:39:00,2024,10,30
377829,2024-10-30 17:41:00,Cerrado,MG,Minas Gerais,3133402,Itapagipe,-19.74592,-49.45157,AQUA_M-T,0.0,0.00,0.63,8.4,31,2024-10-30,17:41:00,2024,10,30
377830,2024-10-30 17:41:00,Cerrado,MG,Minas Gerais,3170206,Uberlândia,-19.15206,-47.93509,NOAA-20,8.0,1.41,0.01,13.6,31,2024-10-30,17:41:00,2024,10,30
377831,2024-10-30 17:41:00,Cerrado,MG,Minas Gerais,3171006,Vazante,-17.82475,-46.73360,AQUA_M-T,8.0,5.00,0.00,11.0,31,2024-10-30,17:41:00,2024,10,30


In [9]:
df_dados_BDMEP

,ano,mes,data_hora,bioma,sigla_uf,sigla_uf_nome,id_municipio,id_municipio_nome,latitude,longitude,satelite,dias_sem_chuva,precipitacao,risco_fogo,potencia_radiativa_fogo,Data,Hora,Ano,Mes,Dia
0,2025,1,2025-01-18 04:25:00,Cerrado,MG,Minas Gerais,3100203,Abaeté,-19.171250,-45.545000,NOAA-21,2.0,0.00,0.00,0.70,2025-01-18,04:25:00,2025,1,18
1,2025,1,2025-01-20 17:04:00,Cerrado,MG,Minas Gerais,3100203,Abaeté,-19.209980,-45.376830,NOAA-20,3.0,0.45,0.01,5.40,2025-01-20,17:04:00,2025,1,20
2,2025,1,2025-01-21 00:26:00,Cerrado,MG,Minas Gerais,3100203,Abaeté,-19.216299,-45.369301,METOP-B,3.0,0.00,0.04,3.65,2025-01-21,00:26:00,2025,1,21
3,2025,1,2025-01-21 03:27:00,Cerrado,MG,Minas Gerais,3100203,Abaeté,-19.207200,-45.380340,NOAA-21,3.0,0.00,0.03,1.90,2025-01-21,03:27:00,2025,1,21
4,2025,1,2025-01-21 03:51:00,Cerrado,MG,Minas Gerais,3100203,Abaeté,-19.206250,-45.380340,NPP-375D,3.0,0.00,0.04,1.50,2025-01-21,03:51:00,2025,1,21
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
405027,2023,8,2023-08-31 22:26:33,Cerrado,MG,Minas Gerais,3135209,Januária,-15.551200,-44.670100,GOES-16,120.0,0.00,1.00,114.40,2023-08-31,22:26:33,2023,8,31
405028,2023,8,2023-08-31 22:16:33,Cerrado,MG,Minas Gerais,3135209,Januária,-15.551200,-44.670100,GOES-16,120.0,0.00,1.00,108.60,2023-08-31,22:16:33,2023,8,31
405029,2023,8,2023-08-31 22:06:33,Cerrado,MG,Minas Gerais,3135209,Januária,-15.551200,-44.670100,GOES-16,120.0,0.00,1.00,101.30,2023-08-31,22:06:33,2023,8,31
405030,2023,8,2023-08-31 22:46:33,Cerrado,MG,Minas Gerais,3135209,Januária,-15.551200,-44.670100,GOES-16,120.0,0.00,1.00,94.10,2023-08-31,22:46:33,2023,8,31


In [10]:
# Ordenaçãon temporal
df_dados_BDMEP = df_dados_BDMEP.sort_values(by=["id_municipio", "data_hora"])

# Garantir datetime
df_dados_BDMEP["data_hora"] = pd.to_datetime(df_dados_BDMEP["data_hora"])

# Intensidade média do fogo
df_dados_BDMEP["frp_media_municipio"] = df_dados_BDMEP.groupby(
    ["id_municipio"]
)["potencia_radiativa_fogo"].transform("mean")

# Intensidade máxima (eventos extremos)
df_dados_BDMEP["frp_max"] = df_dados_BDMEP.groupby(
    ["id_municipio"]
)["potencia_radiativa_fogo"].transform("max")

# Índice de intensidade normalizado
df_dados_BDMEP["frp_norm"] = (
    (df_dados_BDMEP["potencia_radiativa_fogo"] - df_dados_BDMEP["potencia_radiativa_fogo"].min()) /
    (df_dados_BDMEP["potencia_radiativa_fogo"].max() - df_dados_BDMEP["potencia_radiativa_fogo"].min())
)

# Indice de seca simples
df_dados_BDMEP["indice_seca"] = df_dados_BDMEP["dias_sem_chuva"] / (df_dados_BDMEP["precipitacao"] + 1)


In [11]:
df_dados_BDMEP

,ano,mes,data_hora,bioma,sigla_uf,sigla_uf_nome,id_municipio,id_municipio_nome,latitude,longitude,...,potencia_radiativa_fogo,Data,Hora,Ano,Mes,Dia,frp_media_municipio,frp_max,frp_norm,indice_seca
25786,2022,4,2022-04-26 16:56:00,Cerrado,MG,Minas Gerais,3100104,Abadia dos Dourados,-18.26171,-47.43314,...,2.930381,2022-04-26,16:56:00,2022,4,26,13.239462,189.166667,0.000793,3.290078
25787,2022,4,2022-04-26 16:56:00,Cerrado,MG,Minas Gerais,3100104,Abadia dos Dourados,-18.26036,-47.42374,...,2.930227,2022-04-26,16:56:00,2022,4,26,13.239462,189.166667,0.000793,3.290720
26983,2022,7,2022-07-08 16:39:00,Cerrado,MG,Minas Gerais,3100104,Abadia dos Dourados,-18.55436,-47.30458,...,2.746015,2022-07-08,16:39:00,2022,7,8,13.239462,189.166667,0.000743,4.058272
28173,2022,8,2022-08-12 17:02:00,Cerrado,MG,Minas Gerais,3100104,Abadia dos Dourados,-18.49137,-47.37548,...,2.562726,2022-08-12,17:02:00,2022,8,12,13.239462,189.166667,0.000694,4.821974
29144,2022,9,2022-09-03 16:20:00,Cerrado,MG,Minas Gerais,3100104,Abadia dos Dourados,-18.16959,-47.43729,...,2.413169,2022-09-03,16:20:00,2022,9,3,13.239462,189.166667,0.000653,5.445129
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
385936,2024,10,2024-10-03 16:07:00,Mata Atlântica,MG,Minas Gerais,3172202,Wenceslau Braz,-22.55068,-45.37028,...,15.300000,2024-10-03,16:07:00,2024,10,3,8.822222,22.500000,0.004141,9.166667
386142,2024,10,2024-10-03 17:00:00,Mata Atlântica,MG,Minas Gerais,3172202,Wenceslau Braz,-22.54869,-45.37294,...,22.500000,2024-10-03,17:00:00,2024,10,3,8.822222,22.500000,0.006090,9.166667
8255,2024,11,2024-11-28 16:34:00,Mata Atlântica,MG,Minas Gerais,3172202,Wenceslau Braz,-22.57820,-45.38521,...,5.200000,2024-11-28,16:34:00,2024,11,28,8.822222,22.500000,0.001407,0.514359
10489,2024,11,2024-11-28 16:34:00,Mata Atlântica,MG,Minas Gerais,3172202,Wenceslau Braz,-22.58161,-45.38449,...,2.800000,2024-11-28,16:34:00,2024,11,28,8.822222,22.500000,0.000758,1.000000


In [12]:
df_dados_BDMEP.to_parquet(config_path.FEATURES_DIRECTORY_PATH / "dados_BDMEP.parquet")

## Desenvolvimento de Indicadores

### Seleção dos Atributos Preditores e Atributos Alvos